In [0]:
%pip install requests

In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit


# Fetch data from REST Countries API

In [0]:
print("Fetching data from REST Countries API...")
response = requests.get(
    "https://restcountries.com/v3.1/all",
    params={"fields": "name,cca2,region,subregion,population,area,capital,currencies,languages,timezones"},
    timeout=30
)
print(f"Status code: {response.status_code}")
response.raise_for_status()
countries = response.json()
print(f"Fetched {len(countries)} countries")

#Parse the JSON response

In [0]:
records = []
for c in countries:
    records.append({
        "country_code":  c.get("cca2", ""),
        "country_name":  c.get("name", {}).get("common", ""),
        "official_name": c.get("name", {}).get("official", ""),
        "region":        c.get("region", ""),
        "subregion":     c.get("subregion", ""),
        "population":    c.get("population", 0),
        "area_km2":      c.get("area", 0.0),
        "capital":       c.get("capital", [None])[0] if c.get("capital") else None,
        "currencies":    ", ".join(c.get("currencies", {}).keys()),
        "languages":     ", ".join(c.get("languages", {}).values()) if c.get("languages") else None,
        "timezones":     ", ".join(c.get("timezones", [])),
    })

#Convert to Spark DataFrame and write to Bronze

In [0]:
pdf = pd.DataFrame(records)
df = (
    spark.createDataFrame(pdf)
    .withColumn("ingest_at", current_timestamp())
    .withColumn("source", lit("restcountries_api"))
)

df.write.mode("overwrite").format("delta").saveAsTable("workspace.bronze.api_countries")
print(f"Done — {df.count()} rows loaded into workspace.bronze.api_countries")